<a href="https://colab.research.google.com/github/anyuanay/medium/blob/main/src/build_agent_DSPy_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build an LLM-based AI Agent with DSPy

- DSPy is short for Declarative Self-improving Python.
- DSPy's website as for June 2025: https://dspy.ai/
- DSPy is a way to build AI programs faster and more flexible.
- DSPy lets you use clear and structured building blocks.
- DSPy helps you create anything from basic AI tools to complex systems like chatbots or search engines powered by AI.
- You can easily switch between different AI models or techniques without rewriting everything.

In this tutorial, we will build a gemini-based AI agent that help a researcher find arxiv papers on a topic.



### Let us build a simple research assistant agent that can do the following:

- Find a list of paper ids from arxiv based on a topic.
- Retrieve the authors, titles, summaries, and download links of these papers.
- We will build it from dspy.ReAct module.

# Install Packages

## Install DSPy

In [ ]:
!pip install -qU dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and plat

## Install arxiv

In [ ]:
!pip install -q arxiv

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 5.6 MB/s eta 0:00:00


# Import Libraries

In [ ]:
import arxiv
import json
import os
from typing import List

# Define Data Storage
- We will define tools (or functions) to store and retrieve paper information. The paper information can be stored in a database or a file system. In this tutorial, we will simply use JSON files to store the paper information in the local file system.
- If you run this notebook in Colab, you may want to change the following local path to a path in your Drive. Otherwise, all the stored information will be wiped out when the notebook is disconnected.

In [ ]:
PAPER_INFO_PATH = "arxiv_papers"

# Define Tools
We need to prepare a list of tools so that the agent can behave like a human researcher:

- arxiv_search: Search for papers based on a topic.
- arxiv_get_paper_details: Retrieve detailed information about specific paper(s) given their arXiv ID(s).

## Define and test the function `arxiv_search()`:

In [ ]:
def arxiv_search(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information in a location.

    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)

    Returns:
        List of paper IDs found in the search
    """

    # Use arxiv to find the papers
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)

    # Create directory for this topic
    path = os.path.join(PAPER_INFO_PATH, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)

    file_path = os.path.join(path, "papers_info.json")

    # Try to load existing papers info
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    # Process each paper and add to papers_info
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info

    # Save updated papers_info to json file
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)

    print(f"Results are saved in: {file_path}")

    return paper_ids

In [ ]:
# Test
arxiv_search("dspy for agents")

## Define and test the function `arxiv_get_paper_details()`:

In [ ]:
def arxiv_get_paper_details(paper_id: str) -> str:
    """
    Search for information about a specific paper in the direcotry containing paper information

    Args:
        paper_id: The ID of the paper to look for

    Returns:
        JSON string with paper information if found, error message if not found
    """

    for item in os.listdir(PAPER_INFO_PATH):
        item_path = os.path.join(PAPER_INFO_PATH, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue

    return f"There's no saved information related to paper {paper_id}."

In [ ]:
# Test
print(arxiv_get_paper_details("2312.13382v2"))

{
  "title": "DSPy Assertions: Computational Constraints for Self-Refining Language Model Pipelines",
  "authors": [
    "Arnav Singhvi",
    "Manish Shetty",
    "Shangyin Tan",
    "Christopher Potts",
    "Koushik Sen",
    "Matei Zaharia",
    "Omar Khattab"
  ],
  "summary": "Chaining language model (LM) calls as composable modules is fueling a new way\nof programming, but ensuring LMs adhere to important constraints requires\nheuristic \"prompt engineering\". We introduce LM Assertions, a programming\nconstruct for expressing computational constraints that LMs should satisfy. We\nintegrate our constructs into the recent DSPy programming model for LMs, and\npresent new strategies that allow DSPy to compile programs with LM Assertions\ninto more reliable and accurate systems. We also propose strategies to use\nassertions at inference time for automatic self-refinement with LMs. We report\non four diverse case studies for text generation and find that LM Assertions\nimprove not only

# Create ReAct Agent
Now we can create the ReAct agent via dspy.ReAct. We need to provide a signature to dspy.ReAct to define task, and the inputs and outputs of the agent, and tell it about the tools it can access.

In [ ]:
import dspy

class DSPyArXivResearchAssistant(dspy.Signature):
    """
        You are a research assistant that helps user find relevant papers in arxiv and summarize them.

        You will be given a topic and a number of relevant papers to search. You will decide the right tool to
        find the paper ids in arXiv and retrieve the information of the papers.

        You will summiarze the papers at the end.
    """

    user_request: str = dspy.InputField()
    process_result: str = dspy.OutputField(
        desc=(
                "Message that summarizes the process result, and the information users need, e.g., the \
                main research problems, methodology, research results, and gaps."
            )
        )

In [ ]:
agent = dspy.ReAct(
    DSPyArXivResearchAssistant,
    tools = [
        arxiv_search,
        arxiv_get_paper_details
    ]
)

# Use the Agent
- To interact with the agent, simply provide the request through user_request, and the agent will start doing its job.

- Select a language model and set up the API keys. We are using gemini-2.0-flash-lite here, but you can change to other models. For how to configure the language model, please refer to this guide.

In [ ]:
GOOGLE_API_KEY = 'YOUR GOOGLE API KEY'

In [ ]:
dspy.configure(lm=dspy.LM('gemini/gemini-2.0-flash-lite', api_key=GOOGLE_API_KEY))

In [ ]:
result = agent(user_request="please help me find research progress top 3 papers on DSPy for Agents")
print(result)

Results are saved in: arxiv_papers/dspy_for_agents/papers_info.json
Prediction(
    trajectory={'thought_0': 'Okay, I will search for the top 3 papers on DSPy for Agents on arXiv. First, I need to use the arxiv_search tool to find the paper IDs.', 'tool_name_0': 'arxiv_search', 'tool_args_0': {'topic': 'DSPy for Agents', 'max_results': 3}, 'observation_0': ['2310.03714v1', '2312.13382v2', '2504.20965v1'], 'thought_1': 'Now that I have the paper IDs, I will use the arxiv_get_paper_details tool to retrieve the information for each of the papers. I will start with the first paper.', 'tool_name_1': 'arxiv_get_paper_details', 'tool_args_1': {'paper_id': '2310.03714v1'}, 'observation_1': '{\n  "title": "DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines",\n  "authors": [\n    "Omar Khattab",\n    "Arnav Singhvi",\n    "Paridhi Maheshwari",\n    "Zhiyuan Zhang",\n    "Keshav Santhanam",\n    "Sri Vardhamanan",\n    "Saiful Haq",\n    "Ashutosh Sharma",\n    "Thomas

In [ ]:
print(result.process_result)

Here's a summary of the top 3 papers on DSPy for Agents:

1.  **DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines (2310.03714v1):** This paper introduces DSPy, a programming model for developing and optimizing LM pipelines. DSPy abstracts LM pipelines as text transformation graphs, allowing for the learning and optimization of prompting, finetuning, and reasoning techniques. The results show that DSPy programs can outperform standard few-shot prompting and pipelines with expert-created demonstrations.

2.  **DSPy Assertions: Computational Constraints for Self-Refining Language Model Pipelines (2312.13382v2):** This paper introduces LM Assertions, a programming construct for expressing computational constraints that LMs should satisfy. It integrates these constructs into the DSPy programming model to create more reliable and accurate systems. The results show that LM Assertions improve compliance with imposed rules and downstream task performance.

3.  **Aeg

# Interpret the Result
- The result contains the the process_result as required by the user, and a reasoning field that carries the reasoning behind the answer. In addition, it has a trajectory field which contains:

    - Reasoning (thought) at each step Tools picked by LM at each step Arguments for tool calling, determined by LM at each step.
    - Tool execution results at each step Behind scene, the dspy.ReAct is executing a loop, which accumulates tool call information along with the task description, and send to the LM until hits max_iters or the LM decides to wrap up.
    - To better interpret the process, let's use dspy.inspect_history() to see what's happening inside each step.

In [ ]:
dspy.inspect_history(n=10)





[2025-06-15T22:09:21.544952]

System message:

Your input fields are:
1. `user_request` (str): 
2. `trajectory` (str):
Your output fields are:
1. `next_thought` (str): 
2. `next_tool_name` (Literal['arxiv_search', 'arxiv_get_paper_details', 'finish']): 
3. `next_tool_args` (dict[str, Any]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## user_request ## ]]
{user_request}

[[ ## trajectory ## ]]
{trajectory}

[[ ## next_thought ## ]]
{next_thought}

[[ ## next_tool_name ## ]]
{next_tool_name}        # note: the value you produce must exactly match (no extra characters) one of: arxiv_search; arxiv_get_paper_details; finish

[[ ## next_tool_args ## ]]
{next_tool_args}        # note: the value you produce must adhere to the JSON schema: {"type": "object", "additionalProperties": true}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are a research assistant that helps user find relevant papers 

# Conclusion
- Congrats on finishing the tutorial! In this tutorial we have seen how to build a research assistant agent with DSPy. The gists are:

    - Define the tools as python function, and add docstring and type hints.
    - Provide the tools to dspy.ReAct along with a signature to define the task.
    - Invoke the dspy.ReAct with the inputs field defined in the signature, and it will start the reasoning and acting loop behind the scene.